- Ficher source : age-insee-2020.xlsx
- Fichier de sorti : stg_gouv_ages_tranches.csv
- Date de création : 12/11/2025
- Dernière modification : 17/11/2025

- Version(s) : 
    - 1 - 12/11/2025 : Nettoyage du fichier et création du csv "insee_age_tranche"
    - 2 - 14/11/2025 : Renommage du fichier csv de sorti en "gouv_age_tranche"
    - 3 - 17/11/2025 : Renommage du fichier et création du csv "stg_gouv_ages_tranches"

In [1]:
#Librairie(s) utilisée(s)
import pandas as pd

In [2]:
#Création du dataframe à partir du fichier excel
df = pd.read_excel(r'C:\Users\justi\OneDrive\Je-deviens-Data-Analyst\JEDHA\00_Certif\bloc_6\01_data\01_data_bronze\age-insee-2020.xlsx', sheet_name='COM')
df.head()

,INSEE,NOM,EPCI,DEP,REG,F0-2,F3-5,F6-10,F11-17,F18-24,...,H0-2,H3-5,H6-10,H11-17,H18-24,H25-39,H40-54,H55-64,H65-79,H80+
0,1001,L'Abergement-Clémenciat,200069193,D1,R84,13.414280,12.509227,19.214486,37.181506,14.062216,...,18.070033,14.402520,34.539704,40.257329,14.231465,72.497916,81.849108,61.039016,55.240275,18.352851
1,1002,L'Abergement-de-Varey,240100883,D1,R84,2.994218,6.050262,12.232163,11.868718,5.201595,...,2.994218,6.116081,6.953040,22.349436,6.393815,19.540481,37.479209,10.977019,15.687170,8.878739
2,1004,Ambérieu-en-Bugey,240100883,D1,R84,294.667755,245.153009,382.800636,599.105269,680.830755,...,256.303606,289.985107,485.793199,613.181777,669.385044,1542.699092,1238.119870,782.771068,750.039923,252.363550
3,1005,Ambérieux-en-Dombes,200042497,D1,R84,28.000000,33.000000,60.000000,79.000000,50.000000,...,35.000000,36.000000,65.000000,78.000000,51.000000,181.000000,183.000000,124.000000,108.000000,32.000000
4,1006,Ambléon,200040350,D1,R84,0.991228,1.982456,1.982456,0.991228,1.982456,...,1.982456,0.991228,1.982456,0.991228,3.964912,11.894737,10.903509,13.877193,10.903509,1.982456


In [3]:
df.shape

(34980, 26)

In [4]:
df.count()

INSEE          34980
NOM            34980
EPCI           34980
DEP            34980
REG            34980
F0-2           34980
F3-5           34980
F6-10          34980
F11-17         34980
F18-24         34980
F25-39         34980
F40-54         34980
F55-64         34980
F65-79         34980
F80+           34980
Unnamed: 15        0
H0-2           34980
H3-5           34980
H6-10          34980
H11-17         34980
H18-24         34980
H25-39         34980
H40-54         34980
H55-64         34980
H65-79         34980
H80+           34980
dtype: int64

In [5]:
df.count().isna() #Aucune colonnes null

INSEE          False
NOM            False
EPCI           False
DEP            False
REG            False
F0-2           False
F3-5           False
F6-10          False
F11-17         False
F18-24         False
F25-39         False
F40-54         False
F55-64         False
F65-79         False
F80+           False
Unnamed: 15    False
H0-2           False
H3-5           False
H6-10          False
H11-17         False
H18-24         False
H25-39         False
H40-54         False
H55-64         False
H65-79         False
H80+           False
dtype: bool

In [6]:
#Création de la colonne 'MVP' pour cibler les communes cibles et filtrer le dataset dessus
MVP = [35032, 35058, 35196, 35065, 35275, 35144, 35266, 35315, 35278, 35245, 35208, 35353, 35120, 35210, 35352, 35059, 35250, 35079, 35363, 35051, 35055, 35076, 35189, 35066, 35204, 35022, 35351, 35088, 35080, 35131, 35001, 35206, 35081, 35139, 35334, 35216, 35024, 35240, 35039, 35047, 35281, 35180]

df['MVP'] = df['INSEE'].apply(lambda x: 'oui' if x in MVP else 'non')

df['MVP'].describe()

count     34980
unique        2
top         non
freq      34938
Name: MVP, dtype: object

In [7]:
#Création du mask pour filtrer le dataset sur le département de l'Ille-et-Vilaine
mask = (df['DEP'] == 'D35') & (df['MVP'] == 'oui')
df = df[mask]

df.shape

(42, 27)

In [8]:
#Remplacement des '-' par '_' pour l'armonisation des noms de colonnes
df.columns = df.columns.str.replace("-", "_")

#Identification des colonnes 'F' et 'H' pour les ajouter dans des variables
cols_F = sorted([col for col in df.columns if col.startswith("F")])
cols_H = sorted([col for col in df.columns if col.startswith("H")])

#Création d'une boucle pour additionner les colonnes de tranches d'âges identiques dans des nouvelles colonnes
for f_col, h_col in zip(cols_F, cols_H):
    age_label = f_col[1:]
    
    if age_label == "80+":
        df["age_80_max"] = df[f_col] + df[h_col]
        
    else:
        new_col = f"age_{age_label}"
        df[new_col] = df[f_col] + df[h_col]

#Création d'une nouvelle colonne 'age_0_10' pour additionner les tranches d'âges de 0 à 10 ans
df['age_0_10'] = df['age_0_2'] + df['age_3_5'] + df['age_6_10']

#Formatage des types des colonnes 'INSEE' et commençant par 'age'
for col in df.columns:
    if col.startswith("age") or col.startswith("INSEE"):
        df[col] = df[col].astype(int)

df.dtypes

INSEE            int32
NOM             object
EPCI            object
DEP             object
REG             object
F0_2           float64
F3_5           float64
F6_10          float64
F11_17         float64
F18_24         float64
F25_39         float64
F40_54         float64
F55_64         float64
F65_79         float64
F80+           float64
Unnamed: 15    float64
H0_2           float64
H3_5           float64
H6_10          float64
H11_17         float64
H18_24         float64
H25_39         float64
H40_54         float64
H55_64         float64
H65_79         float64
H80+           float64
MVP             object
age_0_2          int32
age_11_17        int32
age_18_24        int32
age_25_39        int32
age_3_5          int32
age_40_54        int32
age_55_64        int32
age_65_79        int32
age_6_10         int32
age_80_max       int32
age_0_10         int32
dtype: object

In [9]:
#Identification des colonnes à garder
keep_columns = ["INSEE","age_0_10","age_11_17","age_18_24","age_25_39","age_40_54","age_55_64","age_65_79","age_80_max"]
df = df.loc[:,keep_columns]

#Renommage de la colonne INSEE
df.rename(columns={"INSEE": "code_geo"}, inplace=True)

df.head()


,code_geo,age_0_10,age_11_17,age_18_24,age_25_39,age_40_54,age_55_64,age_65_79,age_80_max
13186,35001,966,674,508,1171,1524,881,871,265
13205,35022,81,63,44,102,161,91,74,82
13207,35024,1826,1168,946,2284,2504,1748,1612,545
13215,35032,793,507,312,961,1047,424,435,95
13221,35039,335,264,137,365,470,253,180,33


In [ ]:
#Exporte le dataset nettoyé en csv
df.to_csv("stg_gouv_ages_tranches.csv", sep=";", index=False)